# Optional pandas Warm-Up With Progressive Data Views

This optional warm-up practices pandas operations on foundation **Progressive data views**. It is outside the required core module sequence and keeps the canonical model traceable.

In [ ]:
from banking_fraud_lab import build_foundation_progressive_views, generate_minimal_banking_world

In [ ]:
tables = generate_minimal_banking_world(seed=42)
views = build_foundation_progressive_views(tables)

client_relationships = views["foundation_client_relationships"]
alert_lifecycle = views["foundation_alert_lifecycle"]

assert not client_relationships.empty
assert not alert_lifecycle.empty

client_relationships.head()

## Group Client And Banking Relationship Context

Summaries like this help learners inspect Partner, Client, and Banking relationship context before writing more detailed fraud features.

In [ ]:
risk_mix = (
    client_relationships
    .groupby(["institution_name", "risk_rating"], dropna=False)
    .agg(
        client_count=("client_id", "nunique"),
        banking_relationship_count=("banking_relationship_id", "nunique"),
    )
    .reset_index()
    .sort_values(["institution_name", "risk_rating"])
)

assert risk_mix["client_count"].sum() == client_relationships["client_id"].nunique()
risk_mix

## Summarize The Alert Lifecycle

The Alert lifecycle view exposes suspicious activity, alert, case, and outcome context without using protected scenario answer keys.

In [ ]:
alert_summary = (
    alert_lifecycle
    .assign(
        has_case=alert_lifecycle["case_id"].notna(),
        confirmed_fraud_flag=alert_lifecycle["confirmed_fraud"].fillna(False).astype(bool),
    )
    .groupby(["institution_name", "review_priority"], dropna=False)
    .agg(
        alert_count=("alert_id", "nunique"),
        case_count=("has_case", "sum"),
        confirmed_fraud_count=("confirmed_fraud_flag", "sum"),
        suspected_amount_chf=("suspected_amount_chf", "sum"),
    )
    .reset_index()
    .sort_values(["institution_name", "review_priority"])
)

assert alert_summary["alert_count"].sum() == alert_lifecycle["alert_id"].nunique()
alert_summary